In [1]:
import threading

In [25]:
class Counter:
    def __init__(self):
        self.count = 0
        # add a lock here
        self.lock = threading.Lock()

    def increment(self):
        # make this thread-safe
        with self.lock:
            self.count += 1

    def get(self):
        return self.count

In [26]:
counter = Counter()
ts = [threading.Thread(target=counter.increment) for _ in range(100)]
[t.start() for t in ts];
[t.join() for t in ts];
print(counter.count)

100


In [41]:
type(dict())

True

In [39]:
"/a".strip("/").split("/")

['a']

In [42]:
from collections import defaultdict

In [46]:
d = defaultdict(dict)

In [47]:
d["a"] = defaultdict(dict)

In [51]:
"/".join(["a",'b','c'])

'a/b/c'

In [63]:
for c in "abc":
    print(c)

a
b
c


In [66]:
"abcd"[:2]

'ab'

In [67]:
a=[1,2,3] 
b=["a","b","c"]

c=[(x,y) for x,y in zip(a,b)]

In [69]:
for x,y in c:
    print(x, " ", y)

1   a
2   b
3   c


In [56]:
delete


['a', 'b']

In [60]:
"world"+"hello"

'worldhello'

## Practice Problem: Thread-Safe File System

Implement a thread-safe in-memory file system.

**Methods:**
- `mkdir(path)` — creates a directory at path (and all missing parents); returns `True` on success, `False` if directory already exists
- `create(path, content)` — creates a file; returns `True` on success, `False` if the file already exists or the parent directory doesn't exist
- `read(path)` — returns file content as string, or `None` if path doesn't exist
- `append(path, content)` — appends content to existing file; returns `True` on success, `False` if file doesn't exist
- `delete(path)` — deletes the file; returns `True` on success, `False` if file doesn't exist

**Rules:**
- Paths use `/` separator, always start with `/`
- `mkdir` creates all intermediate directories (so `mkdir("/a/b/c")` works in one call)
- `create` requires the parent directory to already exist
- Must be thread-safe

### Peter's Solution

In [93]:
import threading

class FileSystem:
    def __init__(self):
        self.root = {}
        self.lock = threading.Lock()

    def mkdir(self, path):
        with self.lock:
            parts = path.strip("/").split("/")
            if parts[0]=="":
                return False
            node = self.root
            for p in parts[:-1]:
                node = node.setdefault(p, {"__type__":"dir"})
            if parts[-1] in node:
                return False
            node[parts[-1]] = {"__type__":"dir"}
            return True
        
    def create(self, path, content):
        with self.lock:
            parts = path.strip("/").split("/")
            node = self.root
            for p in parts[:-1]:
                if p not in node:
                    return False
                node = node[p]

            fname = parts[-1]
            print(fname)
            if fname in node:
                return False

            node[fname] = {"__type__":"file", "__content__":content}
            return True

    def read(self, path):
        with self.lock:
            parts = path.strip("/").split("/")
            node = self.root
            for p in parts[:-1]:
                if p not in node:
                    return None
                node = node[p]

            fname = parts[-1]
            if fname not in node:
                return None

            return str(node[fname]["__content__"])

    def append(self, path, content):
        with self.lock:
            parts = path.strip("/").split("/")
            node = self.root
            for p in parts[:-1]:
                if p not in node:
                    return False
                node = node[p]

            fname = parts[-1]
            if fname not in node:
                return False

            node[fname]["__content__"] += content
            return True

    def delete(self, path):
        #  deletes the file; returns True on success, False if file doesn't exist
        with self.lock:
            parts = path.strip("/").split("/")
            node = self.root
            for p in parts[:-1]:
                if p not in node:
                    return False
                node = node[p]

            fname = parts[-1]
            if fname not in node:
                return False

            node.pop(fname)
            return True

In [88]:
"/a".strip("/").split("/")

['a']

In [89]:
# Reference solution
import threading

class FileSystem:
    def __init__(self):
        self.root = {}
        self.lock = threading.Lock()

    def _traverse(self, parts):
        node = self.root
        for p in parts:
            if p not in node:
                return None
            node = node[p]
        return node

    def mkdir(self, path):
        parts = path.strip("/").split("/")
        with self.lock:
            node = self.root
            already_exists = True
            for p in parts:
                if p not in node:
                    node[p] = {}
                    already_exists = False
                node = node[p]
            return not already_exists

    def create(self, path, content):
        parts = path.strip("/").split("/")
        with self.lock:
            parent = self._traverse(parts[:-1])
            if parent is None:
                return False
            last = parts[-1]
            if last in parent and "__content__" in parent[last]:
                return False
            parent.setdefault(last, {})["__content__"] = content
            return True

    def read(self, path):
        parts = path.strip("/").split("/")
        with self.lock:
            node = self._traverse(parts)
            if node is None:
                return None
            return node.get("__content__")

    def append(self, path, content):
        parts = path.strip("/").split("/")
        with self.lock:
            node = self._traverse(parts)
            if node is None or "__content__" not in node:
                return False
            node["__content__"] += content
            return True

    def delete(self, path):
        parts = path.strip("/").split("/")
        with self.lock:
            parent = self._traverse(parts[:-1])
            if parent is None:
                return False
            last = parts[-1]
            if last not in parent or "__content__" not in parent[last]:
                return False
            del parent[last]["__content__"]
            return True

In [94]:
# Tests — single-threaded correctness
fs = FileSystem()

# mkdir
assert fs.mkdir("/a/b") == True
assert fs.mkdir("/a/b") == False   # already exists

# create requires parent to exist
assert fs.create("/a/b/file.txt", "hello") == True
assert fs.create("/a/b/file.txt", "other") == False   # duplicate
assert fs.create("/x/y/file.txt", "hi") == False      # /x/y doesn't exist

# read
assert fs.read("/a/b/file.txt") == "hello"
assert fs.read("/a/b/nope.txt") == None

# append
assert fs.append("/a/b/file.txt", " world") == True
assert fs.read("/a/b/file.txt") == "hello world"
assert fs.append("/a/b/nope.txt", "x") == False

# delete
assert fs.delete("/a/b/file.txt") == True
assert fs.read("/a/b/file.txt") == None
assert fs.delete("/a/b/file.txt") == False   # already gone

print("All single-threaded tests passed.")

file.txt
file.txt
All single-threaded tests passed.


In [96]:
# Tests — concurrent correctness
# 100 threads all create unique files; none should collide or corrupt state
import threading

fs3 = FileSystem()
fs3.mkdir("/dir")   # add this line
results = []
lock = threading.Lock()

def worker(i):
    path = f"/dir/file_{i}.txt"
    ok = fs3.create(path, f"content_{i}")
    with lock:
        results.append(ok)

threads = [threading.Thread(target=worker, args=(i,)) for i in range(100)]
[t.start() for t in threads]
[t.join() for t in threads]

assert all(results), "Some creates failed unexpectedly"
assert all(fs3.read(f"/dir/file_{i}.txt") == f"content_{i}" for i in range(100))

print("All concurrent tests passed.")

file_0.txt
file_1.txt
file_2.txt
file_3.txt
file_4.txt
file_5.txt
file_6.txt
file_7.txt
file_8.txt
file_9.txt
file_10.txt
file_11.txt
file_12.txt
file_13.txt
file_14.txt
file_15.txt
file_16.txt
file_17.txt
file_18.txt
file_19.txt
file_20.txt
file_21.txt
file_22.txt
file_23.txt
file_24.txt
file_25.txt
file_26.txt
file_27.txt
file_28.txt
file_29.txt
file_30.txt
file_31.txt
file_32.txt
file_33.txt
file_34.txt
file_35.txt
file_36.txt
file_37.txt
file_38.txt
file_39.txt
file_40.txt
file_41.txt
file_42.txt
file_43.txt
file_44.txt
file_45.txt
file_46.txt
file_47.txt
file_48.txt
file_49.txt
file_50.txt
file_51.txt
file_52.txt
file_53.txt
file_54.txt
file_55.txt
file_56.txt
file_57.txt
file_58.txt
file_59.txt
file_60.txt
file_61.txt
file_62.txt
file_63.txt
file_64.txt
file_65.txt
file_66.txt
file_67.txt
file_68.txt
file_69.txt
file_70.txt
file_71.txt
file_72.txt
file_73.txt
file_74.txt
file_75.txt
file_76.txt
file_77.txt
file_78.txt
file_79.txt
file_80.txt
file_81.txt
file_82.txt
file_83.txt
fi

In [97]:
results

[True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True]